# ESG RL Training -- GRPO + Unsloth

**OpenEnv Hackathon | ESG Compliance & Sustainability Environment**

Trains an LLM agent with **Group Relative Policy Optimization (GRPO)** using verifiable ESG environment rewards.

**Stack:** Unsloth (4-bit LoRA) * TRL (GRPOTrainer) * ESGEnvironment (verifiable reward)  
**Model:** `unsloth/Qwen2.5-0.5B-Instruct` (fast, fits T4)  
**Runtime:** ~45-60 min on Colab T4 GPU (free tier)

| Cell | Purpose | Est. Time |
|------|---------|----------|
| 1 | GPU check | 10 sec |
| 2 | Install dependencies | 3-5 min |
| 3 | Clone repo | 30 sec |
| 4 | Smoke test | 30 sec |
| 5 | Generate dataset | 1 min |
| 6 | GRPO training | ~45 min |
| 7 | Benchmark + plots | 3 min |
| 8 | Push to HuggingFace Hub | 2 min |

In [ ]:
# Cell 1 -- GPU check
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'WARNING: No GPU -- go to Runtime > Change runtime type > T4 GPU')

import torch
assert torch.cuda.is_available(), 'STOP: Switch to T4 GPU in Runtime menu first!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('Ready to train!')

In [ ]:
# Cell 2 -- Install dependencies (~5 min)
# llm-blender is required by trl's judges module (grpo_trainer dependency)
!pip install -q pydantic openai pyyaml datasets
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q trl peft accelerate bitsandbytes huggingface_hub
!pip install -q llm-blender
print('All dependencies installed.')

# Verify TRL imports correctly
from trl import GRPOConfig, GRPOTrainer
print('TRL GRPOTrainer import: OK')

In [ ]:
# Cell 3 -- Clone repo
import os
REPO_DIR = '/content/open_ENV'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/TharunBabu-05/OPEN-ENV.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repository already present -- pulled latest.')

%cd {REPO_DIR}
!python -c "from env import ESGEnvironment; from tasks import TASKS; print('OK -- tasks:', list(TASKS.keys()))"

In [ ]:
# Cell 4 -- Smoke test (validates pipeline without training)
!python train_rl.py --smoke_test
print('If SMOKE TEST PASSED above -- proceed to Cell 5.')

In [ ]:
# Cell 5 -- Generate training dataset (~1 min)
!python dataset_builder.py --episodes 5 --output data/esg_prompts.jsonl

import json
with open('data/esg_prompts.jsonl') as f:
    lines = f.readlines()
print(f'Dataset: {len(lines)} prompts | Tasks: {set(json.loads(l)["task_id"] for l in lines)}')

In [ ]:
# Cell 6 -- GRPO training (~45 min on T4)
# Uses fast config: Qwen2.5-0.5B, 150 steps, basic_compliance task
!python train_rl.py \
    --config train_config_fast.yaml \
    --output_dir outputs/esg_rl_trained

import os
adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    print('\nTraining complete!')
    print('Adapter files:', os.listdir(adapter))
else:
    print('ERROR: Adapter not found. Check training logs above.')

In [ ]:
# Cell 7 -- Benchmark: baseline vs trained model (~3 min)
import os, json

# Heuristic baseline (always run)
!python benchmark.py --mode heuristic --seeds 42 43 44 --output results/baseline_heuristic.json

# Trained model benchmark
adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    !python benchmark.py --mode llm --model_path {adapter} --seeds 42 43 44 --output results/trained.json
    !python plot_results.py \
        --baseline results/baseline_heuristic.json \
        --trained   results/trained.json \
        --output_dir results
else:
    !python plot_results.py --baseline results/baseline_heuristic.json --output_dir results

# Print score comparison
baseline = json.load(open('results/baseline_heuristic.json'))
print(f'Heuristic baseline:  {baseline["overall_mean_score"]:.3f}')
if os.path.exists('results/trained.json'):
    trained = json.load(open('results/trained.json'))
    delta = trained['overall_mean_score'] - baseline['overall_mean_score']
    print(f'Trained model:       {trained["overall_mean_score"]:.3f}')
    print(f'Delta:               {delta:+.3f}')

# Display plots inline
from IPython.display import Image, display
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    path = f'results/{plot}'
    if os.path.exists(path):
        display(Image(path))

In [ ]:
# Cell 8 -- Push trained model + evidence to HuggingFace Hub
import os, getpass
from huggingface_hub import login, HfApi, create_repo

# --- CONFIGURATION ---
HF_TOKEN    = getpass.getpass('Paste your HF Write token (hf_...): ')
HF_USERNAME = 'tharun5054'                         # confirmed from HF profile
MODEL_REPO  = f'{HF_USERNAME}/esg-rl-agent-grpo'   # public model repo
ADAPTER_DIR = 'outputs/esg_rl_trained/lora_adapter'

# --- VALIDATION ---
assert HF_TOKEN.startswith('hf_'), 'Token must start with hf_'
assert os.path.exists(ADAPTER_DIR), f'Adapter not found at {ADAPTER_DIR} -- did Cell 6 complete?'

# --- LOGIN ---
login(token=HF_TOKEN, add_to_git_credential=False)
print(f'Logged in. Pushing to: https://huggingface.co/{MODEL_REPO}')

# --- CREATE REPO ---
create_repo(MODEL_REPO, repo_type='model', exist_ok=True, private=False)

# --- PUSH LORA ADAPTER ---
api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=MODEL_REPO,
    repo_type='model',
    commit_message='Add ESG RL GRPO trained LoRA adapter',
)
print(f'[OK] Adapter pushed -> https://huggingface.co/{MODEL_REPO}')

# --- PUSH BENCHMARK PLOTS AS EVIDENCE ---
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    plot_path = f'results/{plot}'
    if os.path.exists(plot_path):
        api.upload_file(
            path_or_fileobj=plot_path,
            path_in_repo=f'results/{plot}',
            repo_id=MODEL_REPO,
            repo_type='model',
        )
        print(f'[OK] Plot pushed: {plot}')

# --- PUSH BENCHMARK JSON ---
for result_file in ['results/baseline_heuristic.json', 'results/trained.json']:
    if os.path.exists(result_file):
        api.upload_file(
            path_or_fileobj=result_file,
            path_in_repo=result_file,
            repo_id=MODEL_REPO,
            repo_type='model',
        )
        print(f'[OK] Results pushed: {result_file}')

print(f'\nDone!')
print(f'Model URL: https://huggingface.co/{MODEL_REPO}')